# FlowCast — EDA of the joined river + weather data

A simple first look at the data before modeling:

1. Load and join
2. First rows of the dataset
3. What each column means
4. Completeness (missing data)
5. Basic statistics
6. Outliers to be wary of
7. Correlation matrix
8. Summary

**Where the data comes from:** the newest copies in **S3** (the same files every teammate and
the training job use). To use your local `data/` folder instead, set `SOURCE = "local"` in the
next cell. Only needs pandas — no plotting libraries.

**How the join works:** each USGS gauge sits inside one Open-Meteo weather grid square
(`site_weather_cell_map.csv`), so every gauge-day gets the weather of its own square on that
date. This uses the same loading code as `build_training_table.py`, so the EDA looks at exactly
the data the model will use.

In [ ]:
import os, sys
from pathlib import Path

# Run from the repo root no matter where Jupyter was started.
root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root))

SOURCE = "s3"   # "s3" = newest copies in the S3 bucket; "local" = the data/ folder on this computer
os.environ["FLOWCAST_DATA_SOURCE"] = SOURCE

import numpy as np
import pandas as pd
from src.flowcast.storage import read_dataset
from src.flowcast.model.build_training_table import load_river, add_percentiles, add_weather

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:,.2f}".format)

## 1. Load and join

In [ ]:
river = add_percentiles(load_river())
df = add_weather(river)
sites = read_dataset("usgs_active_sites", dtype={"monitoring_location_id": str})
df = df.merge(sites[["monitoring_location_id", "monitoring_location_name", "county_name", "drainage_area"]],
              on="monitoring_location_id", how="left")

COLUMNS = ["monitoring_location_id", "monitoring_location_name", "county_name", "date",
           "streamflow_cfs", "gauge_height_ft", "level_source", "level_value", "level_pct",
           "cell_id", "precipitation_sum", "rain_sum", "snowfall_sum", "precipitation_hours",
           "et0_fao_evapotranspiration", "temperature_2m_max", "temperature_2m_min", "drainage_area"]
df = df[COLUMNS].sort_values(["monitoring_location_id", "date"]).reset_index(drop=True)

print(f"Rows (one per gauge per day): {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Gauges: {df['monitoring_location_id'].nunique()}  |  Weather grid squares: {df['cell_id'].nunique()}")
print(f"Dates: {df['date'].min().date()} to {df['date'].max().date()}")

  reading s3://flowcast-pred-app-data/raw/usgs/daily_values/2026-09-27/usgs_daily_2020_present.csv
  Dropped 6 site(s) with under 365 days of training history.
  reading s3://flowcast-pred-app-data/raw/openmeteo/site_cell_map/2026-09-27/site_weather_cell_map.csv
  reading s3://flowcast-pred-app-data/raw/openmeteo/daily_weather/2026-09-27/weather_daily_2020_present.csv
  Days with no matching weather: 0.0%
  reading s3://flowcast-pred-app-data/raw/usgs/active_sites/2026-09-27/usgs_active_sites.csv
Rows (one per gauge per day): 236,352
Columns: 18
Gauges: 96  |  Weather grid squares: 55
Dates: 2020-01-01 to 2026-09-27


## 2. First rows of the dataset

In [ ]:
df.head(10)

,monitoring_location_id,monitoring_location_name,county_name,date,streamflow_cfs,gauge_height_ft,level_source,level_value,level_pct,cell_id,precipitation_sum,rain_sum,snowfall_sum,precipitation_hours,et0_fao_evapotranspiration,temperature_2m_max,temperature_2m_min,drainage_area
0,USGS-02118000,"SOUTH YADKIN RIVER NEAR MOCKSVILLE, NC",Rowan County,2020-01-01,378.00,3.18,streamflow,378.00,63.08,35.850_-80.650,0.00,0.00,0.00,0.00,0.06,50.40,35.20,306.00
1,USGS-02118000,"SOUTH YADKIN RIVER NEAR MOCKSVILLE, NC",Rowan County,2020-01-02,348.00,3.06,streamflow,348.00,57.64,35.850_-80.650,0.25,0.25,0.00,10.00,0.02,48.70,35.20,306.00
2,USGS-02118000,"SOUTH YADKIN RIVER NEAR MOCKSVILLE, NC",Rowan County,2020-01-03,561.00,3.78,streamflow,561.00,80.87,35.850_-80.650,0.98,0.98,0.00,17.00,0.01,53.60,42.70,306.00
3,USGS-02118000,"SOUTH YADKIN RIVER NEAR MOCKSVILLE, NC",Rowan County,2020-01-04,"1,350.00",6.00,streamflow,"1,350.00",95.05,35.850_-80.650,0.16,0.16,0.00,9.00,0.05,59.70,39.90,306.00
4,USGS-02118000,"SOUTH YADKIN RIVER NEAR MOCKSVILLE, NC",Rowan County,2020-01-05,"1,030.00",5.17,streamflow,"1,030.00",92.83,35.850_-80.650,0.00,0.00,0.00,0.00,0.08,49.50,35.90,306.00
5,USGS-02118000,"SOUTH YADKIN RIVER NEAR MOCKSVILLE, NC",Rowan County,2020-01-06,633.00,4.06,streamflow,633.00,84.24,35.850_-80.650,0.00,0.00,0.00,0.00,0.07,55.00,34.60,306.00
6,USGS-02118000,"SOUTH YADKIN RIVER NEAR MOCKSVILLE, NC",Rowan County,2020-01-07,512.00,3.67,streamflow,512.00,77.15,35.850_-80.650,0.16,0.16,0.00,6.00,0.04,50.60,34.80,306.00
7,USGS-02118000,"SOUTH YADKIN RIVER NEAR MOCKSVILLE, NC",Rowan County,2020-01-08,454.00,3.47,streamflow,454.00,72.91,35.850_-80.650,0.00,0.00,0.00,0.00,0.09,55.10,34.00,306.00
8,USGS-02118000,"SOUTH YADKIN RIVER NEAR MOCKSVILLE, NC",Rowan County,2020-01-09,400.00,3.27,streamflow,400.00,66.69,35.850_-80.650,0.00,0.00,0.00,0.00,0.07,50.90,31.60,306.00
9,USGS-02118000,"SOUTH YADKIN RIVER NEAR MOCKSVILLE, NC",Rowan County,2020-01-10,372.00,3.16,streamflow,372.00,61.55,35.850_-80.650,0.01,0.01,0.00,2.00,0.03,56.80,40.10,306.00


## 3. What each column means

| Column | Meaning | Units | Source |
|---|---|---|---|
| `monitoring_location_id` | USGS gauge ID | — | USGS |
| `monitoring_location_name` | Gauge name (river/creek and place) | — | USGS |
| `county_name` | County the gauge is in | — | USGS |
| `date` | Calendar day (Eastern time) | — | — |
| `streamflow_cfs` | Daily average amount of water flowing past the gauge | cubic feet per second | USGS (parameter 00060) |
| `gauge_height_ft` | Daily average height of the water surface at the gauge | feet | USGS (parameter 00065) |
| `level_source` | Which measurement the gauge's score is based on (streamflow if available, else gauge height) | — | our code |
| `level_value` | The value of that measurement for the day | cfs or ft | our code |
| `level_pct` | **River score**: how high the river is vs. that gauge's own 2020–2024 history (50 = typical, 97 = higher than 97% of days) | 0–100 | our code |
| `cell_id` | Open-Meteo weather grid square (~11 km) the gauge sits in | — | our code |
| `precipitation_sum` | Total rain + snow (as water) that day | inches | Open-Meteo |
| `rain_sum` | Rain only | inches | Open-Meteo |
| `snowfall_sum` | Snow that fell | inches | Open-Meteo |
| `precipitation_hours` | Hours with rain or snow that day | hours | Open-Meteo |
| `et0_fao_evapotranspiration` | Water pulled back out of the ground by sun, heat and wind (drying) | inches | Open-Meteo |
| `temperature_2m_max` / `_min` | Daily high / low air temperature | °F | Open-Meteo |
| `drainage_area` | Land area that drains into the gauge (bigger = bigger river) | square miles | USGS |

## 4. Completeness

**By column:** what share of gauge-days have no value. Streamflow and gauge height are
missing for gauges that only measure the other one — that's expected. Weather should be
close to 0%.

In [ ]:
missing = pd.DataFrame({
    "missing_rows": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(1),
})
missing

,missing_rows,missing_%
monitoring_location_id,0,0.00
monitoring_location_name,0,0.00
county_name,0,0.00
date,0,0.00
streamflow_cfs,43789,18.50
gauge_height_ft,10351,4.40
level_source,0,0.00
level_value,7674,3.20
level_pct,7674,3.20
cell_id,0,0.00


**By gauge:** what share of days each gauge has a reading (`level_value`). Gauges with low
coverage started after 2020, stopped, or had long outages.

In [ ]:
has_reading = df[df["level_value"].notna()]
by_gauge = has_reading.groupby(["monitoring_location_id", "monitoring_location_name"]).agg(
    first_reading=("date", "min"), last_reading=("date", "max")).reset_index()
coverage = df.groupby("monitoring_location_id")["level_value"].apply(lambda s: round(s.notna().mean() * 100, 1))
by_gauge["coverage_pct"] = by_gauge["monitoring_location_id"].map(coverage)
by_gauge["first_reading"] = by_gauge["first_reading"].dt.date
by_gauge["last_reading"] = by_gauge["last_reading"].dt.date

for cutoff in (95, 90, 75):
    print(f"Gauges with readings on at least {cutoff}% of days: "
          f"{(by_gauge['coverage_pct'] >= cutoff).sum()} of {len(by_gauge)}")
print("\nThe 10 least complete gauges:")
by_gauge.sort_values("coverage_pct").head(10)

Gauges with readings on at least 95% of days: 89 of 96
Gauges with readings on at least 90% of days: 89 of 96
Gauges with readings on at least 75% of days: 92 of 96

The 10 least complete gauges:


,monitoring_location_id,monitoring_location_name,first_reading,last_reading,coverage_pct
19,USGS-02130450,"THOMPSON CREEK AT CHESTERFIELD, SC",2020-10-09,2022-06-27,25.50
88,USGS-021473470,"SOUTH FORK FISHING CREEK BELOW McCONNELLS,SC",2020-01-01,2021-09-30,26.00
90,USGS-021474095,"GREAT FALLS RESERVOIR TAILRACE BL GREAT FALLS, SC",2023-03-28,2026-09-26,51.90
3,USGS-02123629,"LITTLE MOUNTAIN CREEK AT NC HWY 740 AT BADIN, NC",2022-08-19,2026-09-26,60.80
51,USGS-02146211,"IRWIN CR AT STATESVILLE AVE AT CHARLOTTE, NC",2020-01-01,2026-09-26,84.90
12,USGS-0212433550,"ROCKY R AB IRISH BUFFALO CR NR ROCKY RIVER, NC",2020-01-01,2026-09-26,87.70
27,USGS-0214244102,"CATAWBA RIVER BL LOOKOUT SHOALS DAM NR SHARON, NC",2020-01-01,2026-01-07,89.30
53,USGS-02146285,STEWART CREEK AT WEST MOREHEAD ST AT CHARLOTTE...,2020-01-01,2026-09-26,95.60
80,USGS-0214676115,"MCALPINE CREEK AT SR2964 NR CAMP COX, SC",2020-01-01,2026-09-26,96.10
86,USGS-021473415,"FISHING CREEK AT HWY 5 BELOW YORK, SC",2020-01-01,2026-07-01,96.40


## 5. Basic statistics

Typical value (50% = median), spread, and extremes for each measurement.
`5%` and `95%` show the normal range; `min` and `max` show the extremes.

In [ ]:
numeric = ["streamflow_cfs", "gauge_height_ft", "level_pct", "precipitation_sum", "rain_sum",
           "snowfall_sum", "precipitation_hours", "et0_fao_evapotranspiration",
           "temperature_2m_max", "temperature_2m_min"]
stats = df[numeric].describe(percentiles=[0.05, 0.5, 0.95]).T
stats[["count", "mean", "std", "min", "5%", "50%", "95%", "max"]]

,count,mean,std,min,5%,50%,95%,max
streamflow_cfs,"192,563.00",492.87,"2,196.68",0.00,0.46,14.80,"2,700.00","121,000.00"
gauge_height_ft,"226,001.00",8.14,41.85,-0.66,0.59,1.95,8.30,360.53
level_pct,"228,678.00",46.94,28.88,0.00,3.86,45.24,94.14,100.00
precipitation_sum,"236,256.00",0.12,0.30,0.00,0.00,0.00,0.66,5.49
rain_sum,"236,256.00",0.12,0.30,0.00,0.00,0.00,0.66,5.49
snowfall_sum,"236,256.00",0.01,0.10,0.00,0.00,0.00,0.00,6.31
precipitation_hours,"236,256.00",3.36,5.02,0.00,0.00,1.00,15.00,24.00
et0_fao_evapotranspiration,"236,256.00",0.13,0.07,0.01,0.03,0.13,0.24,0.32
temperature_2m_max,"236,256.00",72.30,15.14,23.80,45.70,74.50,93.10,103.00
temperature_2m_min,"236,256.00",53.43,14.79,1.20,28.00,55.50,73.20,80.00


**Why streamflow's numbers look extreme:** the table mixes big rivers and small creeks. Each
gauge's own *typical* flow is below — they differ by a large factor, which is why the model uses
the 0–100 river score instead of raw streamflow.

In [ ]:
typical = df.groupby("monitoring_location_id")["streamflow_cfs"].median().dropna()
typical = typical[typical > 0]
print(f"Smallest typical flow: {typical.min():,.1f} cfs")
print(f"Largest typical flow:  {typical.max():,.0f} cfs  (about {typical.max() / typical.min():,.0f}x larger)")
print("\nEach gauge's median river score (should all be near 50 — the score puts every gauge on one scale):")
print(df.groupby("monitoring_location_id")["level_pct"].median().describe()[["min", "50%", "max"]].round(1).to_string())

Smallest typical flow: 0.2 cfs
Largest typical flow:  5,445 cfs  (about 27,225x larger)

Each gauge's median river score (should all be near 50 — the score puts every gauge on one scale):
min   38.40
50%   45.40
max   61.40


## 6. Outliers to be wary of

Each check below flags values that are **impossible**, **suspicious**, or **extreme but real**.
The last group matters most: floods and hurricanes are *exactly* what the model needs to learn,
so extreme-but-real values must **not** be deleted.

In [ ]:
g = df.groupby("monitoring_location_id")

# Day-over-day jump in streamflow (possible sensor glitch if huge and alone).
prev_flow = g["streamflow_cfs"].shift(1)
jump_ratio = df["streamflow_cfs"] / prev_flow.where(prev_flow > 0)

# The same exact reading 14+ days in a row (possible stuck sensor).
run_id = (df["level_value"] != g["level_value"].shift(1)).cumsum()
run_length = df.groupby(run_id)["level_value"].transform("size")
flatline = df["level_value"].notna() & (run_length >= 14)

checks = {
    # --- impossible: should never happen
    "Negative streamflow":                 ("impossible", df["streamflow_cfs"] < 0),
    "Negative rain/snow":                  ("impossible", df["precipitation_sum"] < 0),
    "Negative evaporation":                ("impossible", df["et0_fao_evapotranspiration"] < 0),
    "Low temp above high temp":            ("impossible", df["temperature_2m_min"] > df["temperature_2m_max"]),
    # --- suspicious: check before using
    "Streamflow exactly 0":                ("suspicious", df["streamflow_cfs"] == 0),
    "Negative gauge height":               ("suspicious", df["gauge_height_ft"] < 0),
    "Flow jumps 20x+ in one day":          ("suspicious", jump_ratio >= 20),
    "Same reading 14+ days in a row":      ("suspicious", flatline),
    "Temperature outside 0-110 °F":        ("suspicious", (df["temperature_2m_max"] > 110) | (df["temperature_2m_min"] < 0)),
    # --- extreme but real: keep these
    "Rain over 4 in. in one day":          ("extreme but real", df["precipitation_sum"] > 4),
    "River score 99+ (top 1% for gauge)":  ("extreme but real", df["level_pct"] >= 99),
    "River score 1 or less (bottom 1%)":   ("extreme but real", df["level_pct"] <= 1),
}
what_to_do = {
    "impossible": "Remove / treat as missing",
    "suspicious": "Look at examples; usually keep",
    "extreme but real": "KEEP - floods/droughts are the signal",
}
outliers = pd.DataFrame([
    {"check": name, "type": kind, "rows_flagged": int(mask.fillna(False).sum()),
     "gauges_affected": df.loc[mask.fillna(False), "monitoring_location_id"].nunique(),
     "what_to_do": what_to_do[kind]}
    for name, (kind, mask) in checks.items()
])
outliers

,check,type,rows_flagged,gauges_affected,what_to_do
0,Negative streamflow,impossible,0,0,Remove / treat as missing
1,Negative rain/snow,impossible,0,0,Remove / treat as missing
2,Negative evaporation,impossible,0,0,Remove / treat as missing
3,Low temp above high temp,impossible,0,0,Remove / treat as missing
4,Streamflow exactly 0,suspicious,99,6,Look at examples; usually keep
5,Negative gauge height,suspicious,497,4,Look at examples; usually keep
6,Flow jumps 20x+ in one day,suspicious,2131,69,Look at examples; usually keep
7,Same reading 14+ days in a row,suspicious,116,3,Look at examples; usually keep
8,Temperature outside 0-110 °F,suspicious,0,0,Look at examples; usually keep
9,Rain over 4 in. in one day,extreme but real,76,57,KEEP - floods/droughts are the signal


**Examples** — look at a few flagged rows before deciding anything. Change `CHECK` to see a
different check from the table above.

How to read them:
- **Streamflow exactly 0** — real for small creeks in dry spells; a problem only on big rivers.
- **Negative gauge height** — can be real: gauge height is measured from a fixed mark, not the riverbed.
- **Flow jumps 20x+** — real after a big storm on a small creek; suspicious if rain was near 0 that day.
- **Same reading 14+ days** — often a stuck sensor or USGS "estimated" values.
- **Rain over 4 inches** — check the dates: late Sept 2024 (Hurricane Helene) should show up.

In [ ]:
CHECK = "Rain over 4 in. in one day"   # <- change to any check name from the table

show = ["date", "monitoring_location_id", "monitoring_location_name", "streamflow_cfs",
        "gauge_height_ft", "level_pct", "precipitation_sum", "temperature_2m_max"]
mask = checks[CHECK][1].fillna(False)
print(f"{CHECK}: {mask.sum():,} rows")
df.loc[mask, show].sort_values("date").head(15)

Rain over 4 in. in one day: 76 rows


,date,monitoring_location_id,monitoring_location_name,streamflow_cfs,gauge_height_ft,level_pct,precipitation_sum,temperature_2m_max
36,2020-02-06,USGS-02118000,"SOUTH YADKIN RIVER NEAR MOCKSVILLE, NC","2,770.00",8.33,98.66,4.44,64.70
76358,2020-02-06,USGS-0214265828,MCDOWELL CR TRIB AT SR2131 NR HICKS CROSSROADS...,40.00,NaN,99.81,4.04,65.50
86206,2020-02-06,USGS-0214269560,"KILLIAN CREEK NEAR MARIPOSA, NC","3,300.00",9.62,99.97,4.04,65.50
105902,2020-02-06,USGS-02144000,"LONG CREEK NEAR BESSEMER CITY, NC","2,030.00",5.48,99.92,4.04,65.80
233926,2020-02-06,USGS-02152474,"FIRST BROAD RIVER AT LAWNDALE, NC","6,860.00",12.12,99.92,4.08,63.90
68972,2020-02-06,USGS-0214264790,CATAWBA R AT RR BRIDGE AB NC 73 AT COWANS FORD...,NaN,5.27,80.86,4.04,65.50
66510,2020-02-06,USGS-0214244102,"CATAWBA RIVER BL LOOKOUT SHOALS DAM NR SHARON, NC","12,900.00",15.84,99.21,4.55,64.10
231464,2020-02-06,USGS-02152100,"FIRST BROAD RIVER NEAR CASAR, NC","3,650.00",8.27,99.86,4.22,63.20
2498,2020-02-06,USGS-02118500,"HUNTING CREEK NEAR HARMONY, NC","5,480.00",11.32,99.86,4.38,58.90
189834,2020-09-17,USGS-0214668150,"MCMULLEN CR ABOVE LINCREST PLACE AT CHARLOTTE, NC",NaN,2.05,97.10,5.09,69.80


## 7. Correlation matrix

Correlation runs from **−1 to +1**: +1 = move up together, −1 = one goes up as the other goes
down, 0 = no relationship. Blue cells are negative, red cells positive.

Same-day weather only weakly matches the river, because rivers react to rain over several days
and dry out over weeks. So two helper columns are added to show that (the training table builds
these properly):

- `rain_last_7d` — total rain over the past 7 days
- `river_score_in_3_days` — the river score 3 days later (**what the model will predict** — look at its row)

In [ ]:
corr_df = df[["level_pct", "precipitation_sum", "precipitation_hours", "snowfall_sum",
              "et0_fao_evapotranspiration", "temperature_2m_max", "temperature_2m_min"]].copy()
corr_df["rain_last_7d"] = g["precipitation_sum"].transform(lambda s: s.rolling(7, min_periods=4).sum())
corr_df["river_score_in_3_days"] = g["level_pct"].shift(-3)
corr_df = corr_df.rename(columns={"level_pct": "river_score_today",
                                  "precipitation_sum": "rain_today",
                                  "et0_fao_evapotranspiration": "evaporation_today"})
corr = corr_df.corr().round(2)

def shade(v):
    """Blue for negative, red for positive, stronger color = stronger relationship."""
    if pd.isna(v):
        return ""
    strength = min(abs(v), 1)
    r, g_, b = (184, 48, 47) if v > 0 else (24, 79, 149)
    mix = lambda c, base=240: int(base + (c - base) * strength)
    text = "white" if strength > 0.6 else "#0b0b0b"
    return f"background-color: rgb({mix(r)},{mix(g_)},{mix(b)}); color: {text}"

try:
    styler = corr.style
    # pandas 2.1+ calls it .map; older versions call it .applymap
    styler = styler.map(shade) if hasattr(styler, "map") else styler.applymap(shade)
    display(styler.format("{:.2f}"))
except (ImportError, AttributeError):
    # Table coloring needs the jinja2 package (pip install jinja2). Without it, show plain numbers.
    print("(Install jinja2 for a colored table: pip install jinja2)")
    display(corr)

(Install jinja2 for a colored table: pip install jinja2)


,river_score_today,rain_today,precipitation_hours,snowfall_sum,evaporation_today,temperature_2m_max,temperature_2m_min,rain_last_7d,river_score_in_3_days
river_score_today,1.00,0.30,0.32,0.02,-0.29,-0.27,-0.19,0.42,0.61
rain_today,0.30,1.00,0.75,0.07,-0.31,-0.05,0.10,0.38,0.14
precipitation_hours,0.32,0.75,1.00,0.11,-0.37,-0.02,0.19,0.32,0.16
snowfall_sum,0.02,0.07,0.11,1.00,-0.09,-0.13,-0.11,0.01,0.01
evaporation_today,-0.29,-0.31,-0.37,-0.09,1.00,0.79,0.63,-0.16,-0.22
temperature_2m_max,-0.27,-0.05,-0.02,-0.13,0.79,1.00,0.93,-0.07,-0.24
temperature_2m_min,-0.19,0.10,0.19,-0.11,0.63,0.93,1.00,0.03,-0.20
rain_last_7d,0.42,0.38,0.32,0.01,-0.16,-0.07,0.03,1.00,0.22
river_score_in_3_days,0.61,0.14,0.16,0.01,-0.22,-0.24,-0.20,0.22,1.00


## 8. Summary — fill in from your results

- **Size:** 23652 rows, 96 gauges, 55 weather grid squares.
- **Completeness:** 89 of 96 gauges have readings on 90%+ of days; weather missing on 0% of rows.
- **Scale:** typical flows range from 0.2 to 5445 cfs (27225× apart) → the model uses the 0–100 river score.
- **Outliers:** 0 impossible values (to remove); suspicious: 2843; extreme-but-real events kept (e.g. Helene, late Sept 2024).
- **Correlations:** Today's river score is the strongest predictor of the score 3 days later (r = 0.61). Rain has a weaker linear correlation (r = 0.14–0.22) because the effect is concentrated in infrequent large events and fades within a few days. Heat and evaporation are negatively related to river levels (r ≈ −0.2), reflecting seasonal drying. High and low temperature are nearly redundant (r = 0.93).